In [1]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec, Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
import os
from sentence_transformers import CrossEncoder

d:\scaleRAG\productionRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hp\AppData\Local\Temp\ipykernel_18308\35860561.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

In [4]:
pc=Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

In [5]:
index_name = "prod-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [6]:
file_path = "D:\ProdRAG\prodRAG\example.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [8]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\hp\AppData\Local\Temp\ipykernel_18308\2687298866.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1341.02it/s]


In [9]:
vectorstore = PineconeVectorStore.from_documents(texts, embedder, index_name=index_name)

In [10]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [11]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [12]:
def chunks_docs(docs):
    return [doc.page_content for doc in docs]

In [13]:
rag_prompt = PromptTemplate.from_template(
    template="""You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [14]:
rag_chain = RunnableParallel(
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_docs),
        
    }
)



In [15]:
result=rag_chain | rag_prompt | llm


In [16]:
op=result.invoke("What is summary of module?")


In [17]:
print(op.content)


Module 2: Cryptography Fundamentals 
Module 3: Consensus Algorithms 
Module 4: Smart Contracts and Ethereum 
Module 5: Blockchain and AI Integration


In [ ]:
questions = [
    "How does the proposed course integrate blockchain technology with Artificial Intelligence, Internet of Things, Cybersecurity, and Data Privacy, and why is this interdisciplinary approach important?",
    
    "Compare the consensus mechanisms Proof of Work (PoW), Proof of Stake (PoS), and Byzantine Fault Tolerance (BFT). In what scenarios would each mechanism be most suitable?",
    
    "Describe the complete workflow for developing and deploying a decentralized application (DApp) using the tools mentioned in the course proposal.",
    
    "Why has a Blockchain-based Secure Voting System been selected as the group project, and which blockchain concepts does it demonstrate?",
    
    "Explain the concept of AI model provenance. How can blockchain improve trust, transparency, and verification in AI systems?",
    
    "Discuss the hardware, software, and networking requirements of the course. Why are these resources necessary for blockchain development and AI integration?",
    
    "Analyze the course evaluation scheme. How does it ensure a balanced assessment of theoretical knowledge, practical implementation, and project-based learning?",
    
    "Explain how blockchain technology enhances security in IoT environments. Discuss its role in securing sensor data, communication networks, and edge devices.",
    
    "Evaluate the future scope of the proposed course. How can it contribute to research opportunities, industry collaborations, and institutional growth?",
    
    "Based on the departmental evaluation summary, propose improvements to the course that would satisfy the recommendations of all participating departments."
]

answers = [
    "The course integrates blockchain with AI through model provenance and verifiable model tracking, with IoT by securing sensor data and communication, with cybersecurity through identity management and threat detection, and with data privacy by ensuring integrity and decentralized storage. This interdisciplinary approach prepares students to solve real-world problems that require knowledge across multiple computing domains.",

    "Proof of Work (PoW) achieves consensus through computational mining but consumes significant energy. Proof of Stake (PoS) selects validators based on the amount of cryptocurrency staked, making it more energy efficient. Byzantine Fault Tolerance (BFT) allows distributed systems to reach agreement even when some nodes behave maliciously or fail. Each mechanism offers different trade-offs in security, scalability, and performance.",

    "The workflow begins by writing smart contracts in Solidity, testing them using Remix IDE or Truffle Suite, deploying them on a private blockchain using Ganache or Ethereum-compatible networks, connecting users through MetaMask, and integrating the smart contracts into a decentralized application (DApp).",

    "A Blockchain-based Secure Voting System demonstrates transparency, immutability, decentralization, voter verification, tamper resistance, and smart contract implementation. It provides students with hands-on experience while solving a real-world security problem.",

    "AI model provenance refers to maintaining an immutable record of AI models, datasets, and training history. Blockchain enables secure tracking of model versions, dataset origins, and modifications, thereby improving trust, reproducibility, auditability, and verification of AI systems.",

    "The course requires computing systems with at least 8 GB RAM, GPU-enabled setups, Truffle Suite, MetaMask, Solidity Compiler, Python, Node.js, Web3.py, Web3.js, Pandas, Scikit-learn, and access to Ethereum test networks such as Goerli and Sepolia. These resources support blockchain simulation, smart contract development, AI integration, and decentralized application deployment.",

    "The evaluation allocates marks across assignments and quizzes (20%), mini-projects (20%), mid-term examinations (20%), final project (30%), and attendance and participation (10%). This structure evaluates conceptual understanding, practical skills, project execution, and continuous engagement.",

    "Blockchain improves IoT security by providing immutable storage for sensor data, protecting communication between IoT devices, securing edge devices, enabling hardware-level authentication, and preventing unauthorized modification of collected data.",

    "The proposal envisions establishing a Blockchain Research Cell, promoting collaborations with organizations such as Polygon, IBM Blockchain, and Hyperledger, encouraging research publications, increasing internship opportunities, and expanding into trustworthy AI and auditable intelligent systems.",

    "The proposal could be enhanced by including open-source blockchain contributions for Computer Science, enterprise deployment and DevOps integration for Information Technology, explainable AI and model verification for AI & Data Science, and real-world IoT hardware interfacing with sensor integration for Electronics and Telecommunication."
]